Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

Load Cleaned Dataset

In [2]:
df = pd.read_csv(
    "../data/processed/cleaned_loan_data.csv",
    low_memory=False
)

print(df.shape)

(1303607, 134)


Creating Working Copy

In [3]:
feature_df = df.copy()

Income to Loan Ration

In [4]:
feature_df['income_to_loan_ratio'] = (
    feature_df['annual_inc']
    /
    feature_df['loan_amnt']
)

In [5]:
feature_df['income_to_loan_ratio'].describe()

count    1.303607e+06
mean     7.273031e+00
std      1.153692e+01
min      0.000000e+00
25%      3.437500e+00
50%      5.000000e+00
75%      8.000000e+00
max      5.833333e+03
Name: income_to_loan_ratio, dtype: float64

Credit Utilization Buckets

In [6]:
feature_df['revol_util_bucket'] = pd.cut(
    feature_df['revol_util'],
    bins=[0, 30, 60, 90, 150],
    labels=[
        'low',
        'moderate',
        'high',
        'very_high'
    ]
)

DTI Buckets

In [7]:
feature_df['dti_bucket'] = pd.cut(
    feature_df['dti'],
    bins=[0, 10, 20, 30, 100],
    labels=[
        'low_dti',
        'moderate_dti',
        'high_dti',
        'very_high_dti'
    ]
)

Employment Length Cleaning

In [8]:
feature_df['emp_length'].value_counts()

emp_length
10+ years    428547
2 years      117820
< 1 year     104550
3 years      104200
1 year        85677
5 years       81623
4 years       78029
unknown       75454
6 years       60933
8 years       59125
7 years       58145
9 years       49504
Name: count, dtype: int64

In [9]:
feature_df['emp_length_clean'] = (
    feature_df['emp_length']
    .astype(str)
    .str.extract('(\d+)')
)

feature_df['emp_length_clean'] = pd.to_numeric(
    feature_df['emp_length_clean'],
    errors='coerce'
)

<>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/tmp/ipykernel_76531/3895695987.py:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  .str.extract('(\d+)')


In [10]:
feature_df['emp_length_clean'] = (
    feature_df['emp_length_clean']
    .fillna(0)
)

Employment Stability Buckets

In [11]:
feature_df['employment_stability'] = pd.cut(
    feature_df['emp_length_clean'],
    bins=[-1, 1, 5, 10, 50],
    labels=[
        'new',
        'short_term',
        'medium_term',
        'long_term'
    ]
)

Credit History Length

In [14]:
feature_df['issue_d'] = pd.to_datetime(
    feature_df['issue_d'],
    format='%b-%Y',
    errors='coerce'
)

feature_df['earliest_cr_line'] = pd.to_datetime(
    feature_df['earliest_cr_line'],
    format='%b-%Y',
    errors='coerce'
)

In [15]:
feature_df['credit_history_years'] = (
    (
        feature_df['issue_d']
        -
        feature_df['earliest_cr_line']
    )
    .dt.days
    / 365
)

Inquiry Intensity

In [16]:
feature_df['inquiry_risk_bucket'] = pd.cut(
    feature_df['inq_last_6mths'],
    bins=[-1, 1, 3, 6, 100],
    labels=[
        'low_inquiry',
        'moderate_inquiry',
        'high_inquiry',
        'very_high_inquiry'
    ]
)

Delinquency Intensity

In [17]:
feature_df['delinquency_risk'] = pd.cut(
    feature_df['delinq_2yrs'],
    bins=[-1, 0, 2, 5, 100],
    labels=[
        'no_delinquency',
        'low_delinquency',
        'moderate_delinquency',
        'high_delinquency'
    ]
)

Loan Amount Buckets

In [18]:
feature_df['loan_size_bucket'] = pd.qcut(
    feature_df['loan_amnt'],
    q=4,
    labels=[
        'small',
        'medium',
        'large',
        'very_large'
    ]
)

Revolving Balance to Income Ratio

In [19]:
feature_df['revol_bal_to_income'] = (
    feature_df['revol_bal']
    /
    feature_df['annual_inc']
)

Installment Burden Ratio

In [20]:
feature_df['installment_to_income'] = (
    (
        feature_df['installment'] * 12
    )
    /
    feature_df['annual_inc']
)

Credit Exposure Score

In [21]:
feature_df['credit_exposure_score'] = (
    feature_df['revol_util'].fillna(0)
    +
    feature_df['dti'].fillna(0)
    +
    feature_df['inq_last_6mths'].fillna(0)
)

Validating New Features

In [22]:
feature_df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,target_default,income_to_loan_ratio,revol_util_bucket,dti_bucket,emp_length_clean,employment_stability,credit_history_years,inquiry_risk_bucket,delinquency_risk,loan_size_bucket,revol_bal_to_income,installment_to_income,credit_exposure_score
0,NaN,NaN,30000,30000,30000.0,36 months,22.35,1151.16,D,D5,Supervisor,5 years,mortgage,100000.0,source verified,2018-12-01,fully paid,n,NaN,unknown,debt_consolidation,Debt consolidation,917xx,CA,30.46,0.0,2012-01-01,0.0,51.0,84.0,11.0,1.0,15603,37.0,19.0,w,Jan-2019,0.0,44.0,1,Joint App,150000.0,23.38,Source Verified,0.0,0.0,472330.0,1.0,3.0,2.0,2.0,2.0,82850.0,75.0,0.0,1.0,9713.0,60.0,42200.0,1.0,1.0,3.0,4.0,42939.0,15181.0,46.9,0.0,0.0,83.0,73.0,23.0,2.0,1.0,23.0,38.0,8.0,33.0,0.0,3.0,4.0,3.0,5.0,10.0,6.0,8.0,4.0,11.0,0.0,0.0,0.0,2.0,89.5,33.3,1.0,0.0,527120.0,98453.0,28600.0,101984.0,52417.0,Jul-2006,0.0,1.0,16.0,25.2,2.0,15.0,0.0,0.0,70.0,N,unknown,unknown,unknown,3.0,110.65,unknown,unknown,unknown,3.0,15.0,unknown,307.65,9346.01,121.35,Cash,N,unknown,unknown,unknown,4200.0,45.0,14.0,0,3.333333,moderate,very_high_dti,5.0,short_term,6.920548,low_inquiry,no_delinquency,very_large,0.156030,0.138139,67.46
1,NaN,NaN,40000,40000,40000.0,60 months,16.14,975.71,C,C4,Assistant to the Treasurer (Payroll),< 1 year,mortgage,45000.0,verified,2018-12-01,fully paid,n,NaN,unknown,credit_card,Credit card refinancing,456xx,OH,50.53,0.0,2009-06-01,0.0,31.0,72.0,18.0,0.0,34971,64.5,37.0,w,Feb-2019,0.0,44.0,1,Joint App,92000.0,35.66,Verified,0.0,0.0,271068.0,2.0,8.0,3.0,4.0,5.0,126749.0,87.0,1.0,1.0,5874.0,75.0,54200.0,4.0,2.0,4.0,5.0,15059.0,14930.0,67.3,0.0,0.0,114.0,70.0,2.0,2.0,1.0,2.0,38.0,9.0,33.0,0.0,7.0,9.0,7.0,7.0,26.0,9.0,10.0,9.0,18.0,0.0,0.0,0.0,4.0,100.0,42.9,0.0,0.0,344802.0,161720.0,45700.0,167965.0,47188.0,Apr-1990,0.0,1.0,32.0,61.1,16.0,19.0,0.0,0.0,35.0,N,unknown,unknown,unknown,3.0,110.65,unknown,unknown

In [23]:
feature_df.isnull().sum().sort_values(
    ascending=False
).head(20)

id                     1303607
member_id              1303607
url                    1303607
revol_util_bucket         6710
dti_bucket                1293
desc                       236
revol_bal_to_income          7
loan_amnt                    0
funded_amnt_inv              0
funded_amnt                  0
emp_title                    0
emp_length                   0
home_ownership               0
term                         0
int_rate                     0
installment                  0
grade                        0
sub_grade                    0
pymnt_plan                   0
loan_status                  0
dtype: int64

Analyzing Engineered Features

In [24]:
feature_df.groupby(
    'dti_bucket'
)['target_default'].mean() * 100

dti_bucket
low_dti          14.861115
moderate_dti     17.941489
high_dti         23.191864
very_high_dti    29.526922
Name: target_default, dtype: float64

In [25]:
feature_df.groupby(
    'employment_stability'
)['target_default'].mean() * 100

employment_stability
new            22.472439
short_term     19.914744
medium_term    19.190893
Name: target_default, dtype: float64

In [26]:
feature_df.groupby(
    'inquiry_risk_bucket'
)['target_default'].mean() * 100

inquiry_risk_bucket
low_inquiry          19.161341
moderate_inquiry     24.673391
high_inquiry         28.037962
very_high_inquiry    22.857143
Name: target_default, dtype: float64

Removing remaining weak columns

In [27]:
drop_cols = [
    'id',
    'member_id',
    'url',
    'emp_title'
]

feature_df = feature_df.drop(
    columns=[
        c for c in drop_cols
        if c in feature_df.columns
    ],
    errors='ignore'
)

Saving Feature Engineered Dataset

In [28]:
feature_df.to_csv(
    "/run/media/aditya-anurag/New Volume/Self/Personal/Project/Credit Risk Intelligence & Default Prediction/data/processed/feature_engineered_loan_data.csv",
    index=False
)

print(feature_df.shape)

(1303607, 142)
